In [1]:
# 1. 라이브러리와 경로 설정

import json
from pathlib import Path

import lightgbm as lgb
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            (path / "preprocessing").is_dir()
            and (path / "modeling").is_dir()
            and (path / ".gitignore").exists()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다."
    )

PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

MODELS_DIR = PROJECT_ROOT / "models"

DATA_PATH = (
    PROCESSED_DIR
    / "model_table_enhanced_v2.csv"
)

PARAMS_PATH = (
    MODELS_DIR
    / "best_params.json"
)

V1_VALID_AUC = 0.900928
V1_VALID_PR_AUC = 0.582177
V1_VALID_LOGLOSS = 0.143877

CATEGORICAL_COLS = [
    "city",
    "registered_via",
    "gender",
]

DROP_COLS = [
    "msno",
    "snapshot",
    "split",
    "is_churn",
]

print("프로젝트 루트:", PROJECT_ROOT)
print("입력 데이터:", DATA_PATH)
print("데이터 존재:", DATA_PATH.exists())

프로젝트 루트: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify
입력 데이터: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\data\processed\model_table_enhanced_v2.csv
데이터 존재: True


In [2]:
# 2. v2 데이터 로드 및 기존 split 구성

df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
)

assert df.shape == (992931, 61)
assert df.isna().sum().sum() == 0
assert df["msno"].duplicated().sum() == 0
assert df["snapshot"].eq(
    "2017-01-31"
).all()

for column in CATEGORICAL_COLS:
    df[column] = df[column].astype(
        "category"
    )

FEATURE_COLS = [
    column
    for column in df.columns
    if column not in DROP_COLS
]

train_df = df[
    df["split"] == "train"
]

valid_df = df[
    df["split"] == "valid"
]

X_train = train_df[FEATURE_COLS]
y_train = train_df["is_churn"]

X_valid = valid_df[FEATURE_COLS]
y_valid = valid_df["is_churn"]

assert len(FEATURE_COLS) == 57
assert X_train.shape == (695051, 57)
assert X_valid.shape == (148940, 57)

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=CATEGORICAL_COLS,
)

valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=CATEGORICAL_COLS,
    reference=train_set,
)

print("전체 shape:", df.shape)
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("입력 피처 수:", len(FEATURE_COLS))
print(
    "Train 이탈률:",
    f"{y_train.mean():.4%}",
)
print(
    "Valid 이탈률:",
    f"{y_valid.mean():.4%}",
)
print("v2 학습 데이터 준비 완료")

전체 shape: (992931, 61)
Train: (695051, 57)
Valid: (148940, 57)
입력 피처 수: 57
Train 이탈률: 6.3923%
Valid 이탈률: 6.3925%
v2 학습 데이터 준비 완료


In [3]:
# 3. v1과 동일한 파라미터로 v2 LightGBM 학습

with open(
    PARAMS_PATH,
    "r",
    encoding="utf-8",
) as file:
    saved_config = json.load(file)

v2_params = {
    "objective": "binary",
    "metric": "average_precision",
    "seed": 42,
    "verbose": -1,
    "feature_pre_filter": False,
    **saved_config["params"],
}

v2_model = lgb.train(
    v2_params,
    train_set,
    num_boost_round=3000,
    valid_sets=[valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
        ),
        lgb.log_evaluation(period=100),
    ],
)

print(
    "Best iteration:",
    v2_model.best_iteration,
)

Training until validation scores don't improve for 100 rounds
[100]	valid's average_precision: 0.5753
[200]	valid's average_precision: 0.580134
[300]	valid's average_precision: 0.581744
[400]	valid's average_precision: 0.582557
[500]	valid's average_precision: 0.582716
[600]	valid's average_precision: 0.582905
Early stopping, best iteration is:
[597]	valid's average_precision: 0.582928
Evaluated only: average_precision
Best iteration: 597


In [4]:
# 4. v1과 v2의 Validation 성능 비교

v2_valid_probabilities = v2_model.predict(
    X_valid,
    num_iteration=v2_model.best_iteration,
)

v2_valid_auc = roc_auc_score(
    y_valid,
    v2_valid_probabilities,
)

v2_valid_pr_auc = average_precision_score(
    y_valid,
    v2_valid_probabilities,
)

v2_valid_logloss = log_loss(
    y_valid,
    v2_valid_probabilities,
)

print("===== Enhanced v2 VALID =====")
print(
    "Best iteration:",
    v2_model.best_iteration,
)
print(
    "AUC:",
    f"{v2_valid_auc:.6f}",
)
print(
    "PR-AUC:",
    f"{v2_valid_pr_auc:.6f}",
)
print(
    "LogLoss:",
    f"{v2_valid_logloss:.6f}",
)

print("\n===== v1 대비 변화 =====")
print(
    "AUC 변화:",
    f"{v2_valid_auc - V1_VALID_AUC:+.6f}",
)
print(
    "PR-AUC 변화:",
    f"{v2_valid_pr_auc - V1_VALID_PR_AUC:+.6f}",
)
print(
    "LogLoss 변화:",
    f"{v2_valid_logloss - V1_VALID_LOGLOSS:+.6f}",
)

if v2_valid_pr_auc > V1_VALID_PR_AUC:
    print("\nv2 채택 후보")
else:
    print("\nv1 유지")

===== Enhanced v2 VALID =====
Best iteration: 597
AUC: 0.900879
PR-AUC: 0.582928
LogLoss: 0.143792

===== v1 대비 변화 =====
AUC 변화: -0.000049
PR-AUC 변화: +0.000751
LogLoss 변화: -0.000085

v2 채택 후보


In [6]:
# 5. Validation에서 F1 기준 최종 임계값 결정

import numpy as np

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

valid_precisions, valid_recalls, thresholds = (
    precision_recall_curve(
        y_valid,
        v2_valid_probabilities,
    )
)

valid_f1_scores = (
    2
    * valid_precisions[:-1]
    * valid_recalls[:-1]
    / (
        valid_precisions[:-1]
        + valid_recalls[:-1]
        + 1e-12
    )
)

best_threshold_index = np.argmax(
    valid_f1_scores
)

final_threshold = float(
    thresholds[best_threshold_index]
)

valid_predictions = (
    v2_valid_probabilities
    >= final_threshold
).astype(int)

valid_precision = precision_score(
    y_valid,
    valid_predictions,
)

valid_recall = recall_score(
    y_valid,
    valid_predictions,
)

valid_f1 = f1_score(
    y_valid,
    valid_predictions,
)

print("===== 최종 Validation 기준 =====")
print(
    "Threshold:",
    f"{final_threshold:.6f}",
)
print(
    "Precision:",
    f"{valid_precision:.6f}",
)
print(
    "Recall:",
    f"{valid_recall:.6f}",
)
print(
    "F1:",
    f"{valid_f1:.6f}",
)

===== 최종 Validation 기준 =====
Threshold: 0.285085
Precision: 0.548291
Recall: 0.559290
F1: 0.553736


In [7]:
# 6. 확정된 v2 모델과 임계값으로 Test 최종 평가

test_df = df[
    df["split"] == "test"
]

X_test = test_df[FEATURE_COLS]
y_test = test_df["is_churn"]

assert X_test.shape == (148940, 57)

test_probabilities = v2_model.predict(
    X_test,
    num_iteration=v2_model.best_iteration,
)

test_predictions = (
    test_probabilities
    >= final_threshold
).astype(int)

test_auc = roc_auc_score(
    y_test,
    test_probabilities,
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities,
)

test_logloss = log_loss(
    y_test,
    test_probabilities,
)

test_precision = precision_score(
    y_test,
    test_predictions,
)

test_recall = recall_score(
    y_test,
    test_predictions,
)

test_f1 = f1_score(
    y_test,
    test_predictions,
)

test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
)

print("===== 최종 Enhanced v2 TEST =====")
print("AUC:", f"{test_auc:.6f}")
print(
    "PR-AUC:",
    f"{test_pr_auc:.6f}",
)
print(
    "LogLoss:",
    f"{test_logloss:.6f}",
)
print(
    "Precision:",
    f"{test_precision:.6f}",
)
print(
    "Recall:",
    f"{test_recall:.6f}",
)
print("F1:", f"{test_f1:.6f}")

print("\nConfusion Matrix:")
print(test_confusion_matrix)

print("\n===== 기존 LightGBM 대비 =====")
print(
    "AUC 변화:",
    f"{test_auc - 0.9011266853:+.6f}",
)
print(
    "PR-AUC 변화:",
    f"{test_pr_auc - 0.5752987190:+.6f}",
)
print(
    "LogLoss 변화:",
    f"{test_logloss - 0.1445043155:+.6f}",
)

===== 최종 Enhanced v2 TEST =====
AUC: 0.903610
PR-AUC: 0.582634
LogLoss: 0.143097
Precision: 0.546177
Recall: 0.563445
F1: 0.554677

Confusion Matrix:
[[134963   4457]
 [  4156   5364]]

===== 기존 LightGBM 대비 =====
AUC 변화: +0.002483
PR-AUC 변화: +0.007336
LogLoss 변화: -0.001407


In [8]:
# 7. 상위 고객 마케팅 지표 계산

def calculate_top_k_metrics(
    y_true,
    probabilities,
    top_fraction,
):
    y_array = np.asarray(y_true)
    probability_array = np.asarray(
        probabilities
    )

    selected_count = int(
        np.ceil(
            len(y_array) * top_fraction
        )
    )

    selected_indices = np.argsort(
        -probability_array
    )[:selected_count]

    selected_targets = y_array[
        selected_indices
    ]

    precision_at_k = (
        selected_targets.mean()
    )

    recall_at_k = (
        selected_targets.sum()
        / y_array.sum()
    )

    base_rate = y_array.mean()

    lift_at_k = (
        precision_at_k / base_rate
    )

    return {
        "selected_count": selected_count,
        "precision": precision_at_k,
        "recall": recall_at_k,
        "lift": lift_at_k,
    }


top_5_metrics = calculate_top_k_metrics(
    y_test,
    test_probabilities,
    0.05,
)

top_10_metrics = calculate_top_k_metrics(
    y_test,
    test_probabilities,
    0.10,
)

selection_rate = test_predictions.mean()

base_churn_rate = y_test.mean()

threshold_lift = (
    test_precision
    / base_churn_rate
)

print("===== 마케팅 활용 지표 =====")
print(
    "전체 이탈률:",
    f"{base_churn_rate:.2%}",
)
print(
    "모델 선정 비율:",
    f"{selection_rate:.2%}",
)
print(
    "임계값 기준 Lift:",
    f"{threshold_lift:.2f}배",
)

print("\n상위 5%")
print(
    "선정 고객:",
    f"{top_5_metrics['selected_count']:,}",
)
print(
    "Precision:",
    f"{top_5_metrics['precision']:.2%}",
)
print(
    "Recall:",
    f"{top_5_metrics['recall']:.2%}",
)
print(
    "Lift:",
    f"{top_5_metrics['lift']:.2f}배",
)

print("\n상위 10%")
print(
    "선정 고객:",
    f"{top_10_metrics['selected_count']:,}",
)
print(
    "Precision:",
    f"{top_10_metrics['precision']:.2%}",
)
print(
    "Recall:",
    f"{top_10_metrics['recall']:.2%}",
)
print(
    "Lift:",
    f"{top_10_metrics['lift']:.2f}배",
)

===== 마케팅 활용 지표 =====
전체 이탈률: 6.39%
모델 선정 비율: 6.59%
임계값 기준 Lift: 8.54배

상위 5%
선정 고객: 7,447
Precision: 61.11%
Recall: 47.80%
Lift: 9.56배

상위 10%
선정 고객: 14,894
Precision: 43.19%
Recall: 67.56%
Lift: 6.76배


In [9]:
# 8. 최종 모델·메타데이터·중요도·프론트 파일 저장

MODEL_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v2.txt"
)

META_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v2_meta.json"
)

IMPORTANCE_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v2_importance.csv"
)

FRONTEND_PATH = (
    PROCESSED_DIR
    / "frontend_customer_predictions.csv"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 모델 저장
v2_model.save_model(
    str(MODEL_PATH)
)


# 최종 피처 중요도 저장
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain_importance": (
        v2_model.feature_importance(
            importance_type="gain"
        )
    ),
    "split_importance": (
        v2_model.feature_importance(
            importance_type="split"
        )
    ),
})

feature_importance = (
    feature_importance
    .sort_values(
        "gain_importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)


# 프론트 표시용 주요 컬럼
frontend_columns = [
    "msno",
    "split",
    "is_churn",
    "city",
    "gender",
    "registered_via",
    "bd_clean",
    "tenure_days",
    "days_to_expire",
    "days_since_last_txn",
    "days_since_last_log",
    "last_is_auto_renew",
    "auto_renew_rate",
    "cancel_rate",
    "txn_count",
    "d7_active_days",
    "d30_active_days",
    "d90_active_days",
    "last_plan_list_price",
    "total_amount_paid",
]

frontend_predictions = (
    test_df[frontend_columns]
    .copy()
    .reset_index(drop=True)
)

frontend_predictions[
    "churn_probability"
] = test_probabilities

frontend_predictions[
    "predicted_churn"
] = test_predictions

frontend_predictions[
    "prediction_correct"
] = (
    frontend_predictions["is_churn"]
    == frontend_predictions[
        "predicted_churn"
    ]
).astype("int8")


# 위험도 백분위: 값이 높을수록 고위험
frontend_predictions[
    "risk_percentile"
] = (
    pd.Series(test_probabilities)
    .rank(
        method="average",
        pct=True,
    )
    .mul(100)
    .round(2)
)

risk_percentile = frontend_predictions[
    "risk_percentile"
]

frontend_predictions[
    "risk_grade"
] = np.select(
    [
        risk_percentile >= 95,
        risk_percentile >= 85,
        risk_percentile >= 70,
    ],
    [
        "critical",
        "high",
        "medium",
    ],
    default="low",
)

frontend_predictions = (
    frontend_predictions
    .sort_values(
        "churn_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)

frontend_predictions.to_csv(
    FRONTEND_PATH,
    index=False,
)


# 최종 메타데이터 저장
metadata = {
    "model_name": (
        "LightGBM enhanced v2"
    ),
    "model_role": (
        "final churn prediction model"
    ),
    "feature_cutoff": "2017-01-31",
    "target": "is_churn",
    "source_table": (
        "model_table_enhanced_v2.csv"
    ),
    "early_stopping_metric": (
        "average_precision"
    ),
    "best_iteration": int(
        v2_model.best_iteration
    ),
    "threshold_source": (
        "Validation F1 maximum"
    ),
    "threshold": float(
        final_threshold
    ),
    "feature_count": len(
        FEATURE_COLS
    ),
    "feature_cols": FEATURE_COLS,
    "categorical_cols": (
        CATEGORICAL_COLS
    ),
    "params": v2_params,
    "valid_auc": float(
        v2_valid_auc
    ),
    "valid_pr_auc": float(
        v2_valid_pr_auc
    ),
    "valid_logloss": float(
        v2_valid_logloss
    ),
    "valid_precision": float(
        valid_precision
    ),
    "valid_recall": float(
        valid_recall
    ),
    "valid_f1": float(
        valid_f1
    ),
    "test_auc": float(test_auc),
    "test_pr_auc": float(
        test_pr_auc
    ),
    "test_logloss": float(
        test_logloss
    ),
    "test_precision": float(
        test_precision
    ),
    "test_recall": float(
        test_recall
    ),
    "test_f1": float(test_f1),
    "test_confusion_matrix": (
        test_confusion_matrix.tolist()
    ),
    "test_selection_rate": float(
        selection_rate
    ),
    "test_threshold_lift": float(
        threshold_lift
    ),
    "top_5_percent": {
        key: float(value)
        if key != "selected_count"
        else int(value)
        for key, value
        in top_5_metrics.items()
    },
    "top_10_percent": {
        key: float(value)
        if key != "selected_count"
        else int(value)
        for key, value
        in top_10_metrics.items()
    },
}

with open(
    META_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("모델:", MODEL_PATH)
print("메타데이터:", META_PATH)
print("중요도:", IMPORTANCE_PATH)
print("프론트 파일:", FRONTEND_PATH)
print(
    "프론트 행 수:",
    f"{len(frontend_predictions):,}",
)
print("\n최종 산출물 저장 완료")

모델: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\lightgbm_enhanced_v2.txt
메타데이터: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\lightgbm_enhanced_v2_meta.json
중요도: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\lightgbm_enhanced_v2_importance.csv
프론트 파일: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\data\processed\frontend_customer_predictions.csv
프론트 행 수: 148,940

최종 산출물 저장 완료
